In [242]:
!pip install pyreadstat

In [243]:
import pandas as pd
import numpy as np


In [244]:

import pyreadstat
import pandas as pd

# Archivo 2019 desde SPSS
anemia_2019, _ = pyreadstat.read_sav("/content/REC44_2019.sav")
hogar_2019,  _ = pyreadstat.read_sav("/content/RECH0_2019.SAV")

# Convertir a string igual que los demás años
anemia_2019 = anemia_2019.astype(str)
hogar_2019  = hogar_2019.astype(str)

# Archivos de anemia 2020-2024
anemia_2020 = pd.read_csv("/content/REC44_2020.csv", encoding="utf-8-sig", sep=",", dtype=str)
anemia_2021 = pd.read_csv("/content/REC44_2021.csv", encoding="utf-8-sig", sep=",", dtype=str)
anemia_2022 = pd.read_csv("/content/REC44_2022.csv", encoding="utf-8-sig", sep=",", dtype=str)
anemia_2023 = pd.read_csv("/content/REC44_2023.csv", encoding="utf-8-sig", sep=",", dtype=str)
anemia_2024 = pd.read_csv("/content/REC44_2024.csv", encoding="utf-8-sig", sep=",", dtype=str)

# Archivos de hogar 2020-2024
hogar_2020 = pd.read_csv("/content/RECH0_2020.csv", encoding="utf-8-sig", sep=";", dtype=str)
hogar_2021 = pd.read_csv("/content/RECH0_2021.csv", encoding="latin-1",   sep=",", dtype=str)
hogar_2022 = pd.read_csv("/content/RECH0_2022.csv", encoding="latin-1",   sep=";", dtype=str)
hogar_2023 = pd.read_csv("/content/RECH0_2023.csv", encoding="utf-8-sig", sep=",", dtype=str)
hogar_2024 = pd.read_csv("/content/RECH0_2024.csv", encoding="utf-8-sig", sep=",", dtype=str)

hogar_2022.columns = hogar_2022.columns.str.replace('ï»¿', '', regex=False).str.strip()

print("Archivos cargados correctamente")
print("Anemia 2019:", anemia_2019.shape)
print("Hogar  2019:", hogar_2019.shape)

Archivos cargados correctamente
Anemia 2019: (21154, 32)
Hogar  2019: (37474, 44)


In [245]:
#Limpiar decimales
# Columnas que vienen como '1.0', '2.0' en el 2019
columnas_limpiar = ['HV024', 'HV025', 'HV040', 'HV005', 'HV022', 'HV001']

for col in columnas_limpiar:
    if col in hogar_2019.columns:
        hogar_2019[col] = hogar_2019[col].str.replace('.0', '', regex=False)

print("Valores HV024 después de limpiar:")
print(hogar_2019['HV024'].unique()[:10])

Valores HV024 después de limpiar:
['1' '2' '3' '4' '5' '6' '7' '8' '9' '10']


In [246]:
# Convertir 2019.0 a 2019 en ambas tablas
anemia_2019['ID1'] = anemia_2019['ID1'].str.replace('.0', '', regex=False)
hogar_2019['ID1']  = hogar_2019['ID1'].str.replace('.0', '', regex=False)

print("ID1 anemia 2019 corregido:", anemia_2019['ID1'].unique())
print("ID1 hogar  2019 corregido:", hogar_2019['ID1'].unique())

ID1 anemia 2019 corregido: ['2019']
ID1 hogar  2019 corregido: ['2019']


In [247]:
# Unir todos los archivos de anemia
df_anemia = pd.concat([anemia_2019, anemia_2020, anemia_2021,
                       anemia_2022, anemia_2023, anemia_2024], ignore_index=True)

# Unir todos los archivos de hogar
df_hogar = pd.concat([hogar_2019, hogar_2020, hogar_2021,
                      hogar_2022, hogar_2023, hogar_2024], ignore_index=True)

print("Tabla anemia:", df_anemia.shape)
print("Tabla hogar: ", df_hogar.shape)

Tabla anemia: (122698, 34)
Tabla hogar:  (225262, 47)


In [248]:
df_anemia['HHID'] = df_anemia['CASEID'].str[:15]

df_final = df_anemia.merge(df_hogar, on=['HHID', 'ID1'], how='left')

print("Tabla final:", df_final.shape)
print("Nulos en HV024:", df_final['HV024'].isna().sum())

print("\nRegistros por año:")
print(df_final.groupby('ID1').size().reset_index(name='Total'))

Tabla final: (122698, 80)
Nulos en HV024: 0

Registros por año:
    ID1  Total
0  2019  21154
1  2020  17242
2  2021  22100
3  2022  21611
4  2023  20840
5  2024  19751


In [249]:
print("Registros por año:")
print(df_final.groupby('ID1').size().reset_index(name='Total'))

df_final.head(5)

Registros por año:
    ID1  Total
0  2019  21154
1  2020  17242
2  2021  22100
3  2022  21611
4  2023  20840
5  2024  19751


,ID1,CASEID,HWIDX,HW1,HW2,HW3,HW4,HW5,HW6,HW7,...,HV005,UBIGEO,NCONGLOME,CODCCPP,NOMCCPP,longitudx,latitudy,LONGITUDX,LATITUDY,NCONGLOME1
0,2019,000100201 2,1.0,8.0,107.0,727.0,6511.0,39.0,10143.0,9519.0,...,142155,010101,7065.0,0001,CHACHAPOYAS,-77.87383,-6.22332,NaN,NaN,NaN
1,2019,000102801 2,1.0,18.0,93.0,790.0,2351.0,-72.0,9726.0,892.0,...,142155,010101,7065.0,0001,CHACHAPOYAS,-77.87383,-6.22332,NaN,NaN,NaN
2,2019,000102801 2,2.0,52.0,182.0,1059.0,5504.0,13.0,10053.0,6540.0,...,142155,010101,7065.0,0001,CHACHAPOYAS,-77.87383,-6.22332,NaN,NaN,NaN
3,2019,000104801 2,1.0,42.0,145.0,963.0,3002.0,-52.0,9792.0,3385.0,...,142155,010101,7065.0,0001,CHACHAPOYAS,-77.87383,-6.22332,NaN,NaN,NaN
4,2019,000113601 2,1.0,8.0,85.0,685.0,4210.0,-20.0,9922.0,6407.0,...,142155,010101,7065.0,0001,CHACHAPOYAS,-77.87383,-6.22332,NaN,NaN,NaN


In [250]:
# Seleccionamos solo las columnas que necesitamos
df_final = df_final[['ID1', 'CASEID', 'HHID', 'HW1', 'HW2', 'HW3',
                      'HW56', 'HW70', 'HW71', 'HW72', 'HW73', 'HW57',
                      'HV024', 'HV025', 'HV040', 'UBIGEO',
                      'HV005', 'HV022', 'HV001']]

print("Columnas seleccionadas:", list(df_final.columns))
print("Filas:", len(df_final))

Columnas seleccionadas: ['ID1', 'CASEID', 'HHID', 'HW1', 'HW2', 'HW3', 'HW56', 'HW70', 'HW71', 'HW72', 'HW73', 'HW57', 'HV024', 'HV025', 'HV040', 'UBIGEO', 'HV005', 'HV022', 'HV001']
Filas: 122698


In [251]:
#Renombramos columnas
df_final = df_final.rename(columns={
    'ID1'   : 'ANO',
    'HW1'   : 'Edad_Meses',
    'HW2'   : 'Peso_kg',
    'HW3'   : 'Talla_cm',
    'HW56'  : 'Hemoglobina',
    'HW70'  : 'Zscore_Talla_Edad',
    'HW71'  : 'Zscore_Peso_Edad',
    'HW72'  : 'Zscore_Peso_Talla',
    'HW73'  : 'Zscore_IMC',
    'HW57'  : 'Nivel_Anemia',
    'HV024' : 'Departamento',
    'HV025' : 'Area_Residencia',
    'HV040' : 'Altitud',
    'HV005' : 'Factor_Expansion',
    'HV022' : 'Estrato',
    'HV001' : 'Conglomerado'
})

print("Columnas renombradas:")
print(list(df_final.columns))
df_final.head(3)

Columnas renombradas:
['ANO', 'CASEID', 'HHID', 'Edad_Meses', 'Peso_kg', 'Talla_cm', 'Hemoglobina', 'Zscore_Talla_Edad', 'Zscore_Peso_Edad', 'Zscore_Peso_Talla', 'Zscore_IMC', 'Nivel_Anemia', 'Departamento', 'Area_Residencia', 'Altitud', 'UBIGEO', 'Factor_Expansion', 'Estrato', 'Conglomerado']


,ANO,CASEID,HHID,Edad_Meses,Peso_kg,Talla_cm,Hemoglobina,Zscore_Talla_Edad,Zscore_Peso_Edad,Zscore_Peso_Talla,Zscore_IMC,Nivel_Anemia,Departamento,Area_Residencia,Altitud,UBIGEO,Factor_Expansion,Estrato,Conglomerado
0,2019,000100201 2,000100201,8.0,107.0,727.0,121.0,63.0,185.0,200.0,195.0,4.0,1,1,2356,010101,142155,3,1
1,2019,000102801 2,000102801,18.0,93.0,790.0,126.0,-69.0,-84.0,-70.0,-61.0,4.0,1,1,2356,010101,142155,3,1
2,2019,000102801 2,000102801,52.0,182.0,1059.0,120.0,9.0,52.0,70.0,71.0,4.0,1,1,2356,010101,142155,3,1


In [252]:
####Comvertimos columnas numéricas
df_final['Edad_Meses'] = pd.to_numeric(df_final['Edad_Meses'], errors='coerce')
df_final['Peso_kg']    = pd.to_numeric(df_final['Peso_kg'],    errors='coerce') / 10
df_final['Talla_cm']   = pd.to_numeric(df_final['Talla_cm'],   errors='coerce') / 10
df_final['Hemoglobina']= pd.to_numeric(df_final['Hemoglobina'],errors='coerce') / 10

df_final['Zscore_Talla_Edad'] = pd.to_numeric(df_final['Zscore_Talla_Edad'], errors='coerce') / 100
df_final['Zscore_Peso_Edad']  = pd.to_numeric(df_final['Zscore_Peso_Edad'],  errors='coerce') / 100
df_final['Zscore_Peso_Talla'] = pd.to_numeric(df_final['Zscore_Peso_Talla'], errors='coerce') / 100
df_final['Zscore_IMC']        = pd.to_numeric(df_final['Zscore_IMC'],        errors='coerce') / 100

df_final['Altitud']          = pd.to_numeric(df_final['Altitud'],          errors='coerce')
df_final['Factor_Expansion'] = pd.to_numeric(df_final['Factor_Expansion'], errors='coerce') / 1000000

print("Conversión lista")

Conversión lista


In [253]:
# Valores muy altos son códigos de "sin dato" en ENDES
df_final['Peso_kg']    = df_final['Peso_kg'].where(df_final['Peso_kg'] < 300, None)
df_final['Talla_cm']   = df_final['Talla_cm'].where(df_final['Talla_cm'] < 300, None)
df_final['Hemoglobina']= df_final['Hemoglobina'].where(df_final['Hemoglobina'] < 30, None)

print("Valores imposibles reemplazados")
df_final[['Peso_kg', 'Talla_cm', 'Hemoglobina']].describe()

Valores imposibles reemplazados


,Peso_kg,Talla_cm,Hemoglobina
count,115818.000000,114436.000000,105108.000000
mean,13.912152,86.939973,11.337598
std,10.135519,13.949523,1.093044
min,1.700000,40.600000,3.400000
25%,10.000000,76.600000,10.700000
50%,12.900000,88.800000,11.400000
75%,15.600000,98.100000,12.100000
max,99.900000,141.700000,17.800000


In [254]:
###Indexando valores
# Nivel de anemia
mapa_anemia = {'1': 'Severa', '2': 'Moderada', '3': 'Leve', '4': 'Sin anemia'}
df_final['Nivel_Anemia'] = df_final['Nivel_Anemia'].map(mapa_anemia)

# Área de residencia
mapa_area = {'1': 'Urbana', '2': 'Rural'}
df_final['Area_Residencia'] = df_final['Area_Residencia'].map(mapa_area)

# Departamento
mapa_deptos = {
    '1':'Amazonas',  '2':'Áncash',      '3':'Apurímac',  '4':'Arequipa',
    '5':'Ayacucho',  '6':'Cajamarca',   '7':'Callao',    '8':'Cusco',
    '9':'Huancavelica', '10':'Huánuco', '11':'Ica',      '12':'Junín',
    '13':'La Libertad', '14':'Lambayeque', '15':'Lima',  '16':'Loreto',
    '17':'Madre de Dios', '18':'Moquegua', '19':'Pasco', '20':'Piura',
    '21':'Puno',     '22':'San Martín', '23':'Tacna',    '24':'Tumbes',
    '25':'Ucayali'
}
df_final['Departamento'] = df_final['Departamento'].map(mapa_deptos)

print("Etiquetas aplicadas")
df_final[['Nivel_Anemia', 'Area_Residencia', 'Departamento']].value_counts().head(10)

Etiquetas aplicadas


Nivel_Anemia  Area_Residencia  Departamento
Sin anemia    Urbana           Lima            7129
                               Callao          2436
                               Ica             2201
                               Tumbes          2169
                               Lambayeque      2112
                               Tacna           2071
                               Piura           1975
                               Arequipa        1851
                               Moquegua        1849
Leve          Urbana           Lima            1807
Name: count, dtype: int64

In [255]:
df_limpio = df_final.dropna(subset=['Edad_Meses', 'Peso_kg', 'Talla_cm'])
df_con_hemoglobina = df_limpio.dropna(subset=['Hemoglobina'])

print("=== RESUMEN FINAL ===")
print("Total registros:       ", len(df_final))
print("Niños medidos:         ", len(df_limpio))
print("Con hemoglobina:       ", len(df_con_hemoglobina))

print("\nNulos restantes:")
print(df_limpio.isnull().sum())

print("\nRegistros por año:")
print(df_limpio.groupby('ANO').size().reset_index(name='Total'))

df_final.to_csv("/content/base_ENDES_consolidada.csv", index=False, encoding='utf-8-sig')
print("\n✓ Archivo guardado")

=== RESUMEN FINAL ===
Total registros:        122698
Niños medidos:          114436
Con hemoglobina:        105027

Nulos restantes:
ANO                      0
CASEID                   0
HHID                     0
Edad_Meses               0
Peso_kg                  0
Talla_cm                 0
Hemoglobina           9409
Zscore_Talla_Edad        0
Zscore_Peso_Edad         0
Zscore_Peso_Talla        0
Zscore_IMC               0
Nivel_Anemia         28248
Departamento             0
Area_Residencia          0
Altitud                  0
UBIGEO                   0
Factor_Expansion         0
Estrato                  0
Conglomerado             0
dtype: int64

Registros por año:
    ANO  Total
0  2019  20565
1  2020  12538
2  2021  21310
3  2022  20917
4  2023  20138
5  2024  18968

✓ Archivo guardado


In [256]:
#Limpiamos nulos
#filas donde el niño no fue medido
df_limpio = df_final.dropna(subset=['Edad_Meses', 'Peso_kg', 'Talla_cm'])

print("Filas antes:", len(df_final))
print("Filas después:", len(df_limpio))
print("Eliminadas:", len(df_final) - len(df_limpio))

Filas antes: 122698
Filas después: 114436
Eliminadas: 8262


In [257]:
# Tabla nueva con niños que se midieron hemoglobina
df_con_hemoglobina = df_limpio.dropna(subset=['Hemoglobina'])

print("Niños medidos:    ", len(df_limpio))
print("Con hemoglobina:  ", len(df_con_hemoglobina))
print("Sin hemoglobina:  ", len(df_limpio) - len(df_con_hemoglobina))

print("\nPor año:")
print(df_con_hemoglobina.groupby('ANO').size().reset_index(name='Total'))

Niños medidos:     114436
Con hemoglobina:   105027
Sin hemoglobina:   9409

Por año:
    ANO  Total
0  2019  18839
1  2020  11431
2  2021  19770
3  2022  19253
4  2023  18331
5  2024  17403


In [258]:
print(df_con_hemoglobina.isnull().sum())

ANO                      0
CASEID                   0
HHID                     0
Edad_Meses               0
Peso_kg                  0
Talla_cm                 0
Hemoglobina              0
Zscore_Talla_Edad        0
Zscore_Peso_Edad         0
Zscore_Peso_Talla        0
Zscore_IMC               0
Nivel_Anemia         18839
Departamento             0
Area_Residencia          0
Altitud                  0
UBIGEO                   0
Factor_Expansion         0
Estrato                  0
Conglomerado             0
dtype: int64


In [259]:
# Donde no hay Nivel_Anemia lo calculamos con la hemoglobina, llenamos vacios
def clasificar_anemia(hb):
    if hb < 7.0:
        return 'Severa'
    elif hb < 10.0:
        return 'Moderada'
    elif hb < 11.0:
        return 'Leve'
    else:
        return 'Sin anemia'

# Aplicar solo donde Nivel_Anemia es nulo
mask = df_con_hemoglobina['Nivel_Anemia'].isna()
df_con_hemoglobina.loc[mask, 'Nivel_Anemia'] = df_con_hemoglobina.loc[mask, 'Hemoglobina'].apply(clasificar_anemia)

print("Nulos restantes en Nivel_Anemia:", df_con_hemoglobina['Nivel_Anemia'].isna().sum())
print("\nDistribución:")
print(df_con_hemoglobina['Nivel_Anemia'].value_counts())

Nulos restantes en Nivel_Anemia: 0

Distribución:
Nivel_Anemia
Sin anemia    69554
Leve          25026
Moderada      10328
Severa          119
Name: count, dtype: int64


In [274]:
df_con_hemoglobina.to_csv(
    '/content/ENDES_Anemia_Hogar_2019_2024_LIMPIO.csv',
    index=False,
    encoding='utf-8-sig'
)

In [275]:
from google.colab import files

files.download('/content/ENDES_Anemia_Hogar_2019_2024_LIMPIO.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>